# MODELO DE REGRESIÓN — Ingreso laboral (ENAHO 2023)

**Target:** ingreso anual del trabajo principal (`i524a1`, la ENAHO lo reporta anual), en logaritmo: `y_reg = log1p(i524a1)`.
**Universo:** 25 232 asalariados ocupados con ingreso declarado (ENAHO 2023).
**Predictores (solo sociodemográficos, sin variables de empleo/ingreso):** edad, sexo, parentesco, nivel educativo, lengua materna, dominio, área y campo de estudio.

**Cómo usarlo:**
1. Archivo → Subir copia en Drive (o abrir en Colab).
2. Entorno de ejecución → Ejecutar todo.
3. La base se lee de la carpeta compartida `CODIGOS_ENAPRES/BASE_ENAHO/2023` (ruta en la celda de Drive).
4. Al final se descarga el modelo entrenado (`regresor_ingreso.joblib`) para el despliegue en Streamlit.


## Importing the libraries

In [ ]:
# Instala lo que no viene por defecto en Colab
!pip install -q pyreadstat joblib

# Cómo importar las librerías
import os
import unicodedata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pyreadstat
import joblib

## 1. Lectura de datos

Montamos Google Drive y cargamos los 3 módulos `.sav` (Miembros del hogar, Educación, Empleo e Ingresos) desde la carpeta compartida.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Ruta de la carpeta compartida (cámbiala si tu Drive la tiene en otro lugar)
RUTA_BASE = '/content/drive/My Drive/CODIGOS_ENAPRES/BASE_ENAHO/2023'
print('Ruta base:', RUTA_BASE, '| Existe:', os.path.isdir(RUTA_BASE))

In [ ]:
def norm(c):
    return unicodedata.normalize('NFKD', str(c)).encode('ascii', 'ignore').decode('ascii').lower()

ARCHIVOS = {
    'mod02': RUTA_BASE + '/mod02/906-Modulo02/Enaho01-2023-200.sav',
    'mod03': RUTA_BASE + '/mod03/906-Modulo03/Enaho01A-2023-300.sav',
    'mod05': RUTA_BASE + '/mod05/906-Modulo05/Enaho01a-2023-500.sav',
}

modulos = {}
for nombre, ruta in ARCHIVOS.items():
    df, _ = pyreadstat.read_sav(ruta)
    df.columns = [norm(c) for c in df.columns]
    modulos[nombre] = df
    print(nombre, df.shape)

### Merge y filtros (spec: 86 654 → 59 447 → 25 244 → 25 232)

1. Inner join por llave persona `(mes, conglome, vivienda, hogar, codperso)`
2. PEA ocupada (`ocupinf` no nulo)
3. Asalariados con ingreso (`i524a1 > 0`)
4. Sin nulos en predictores

In [ ]:
LLAVES = ['mes', 'conglome', 'vivienda', 'hogar', 'codperso']

sub02 = modulos['mod02'][LLAVES + ['p203', 'p207', 'p208a', 'estrato', 'dominio']]
sub03 = modulos['mod03'][LLAVES + ['p301a', 'p300a', 'p301a1']]
sub05 = modulos['mod05'][LLAVES + ['ocupinf', 'i524a1']]

paso1 = sub05.merge(sub02, on=LLAVES, how='inner', validate='one_to_one')
paso1 = paso1.merge(sub03, on=LLAVES, how='inner', validate='one_to_one')
paso2 = paso1[paso1['ocupinf'].notna()].copy()
paso3 = paso2[paso2['i524a1'] > 0].copy()
print('1. Join completo:', len(paso1))
print('2. PEA ocupada :', len(paso2))
print('3. Con ingreso :', len(paso3))

## 2. Transformaciones de datos

- `area`: urbano (1) si estrato ≤ 5, rural (0) si ≥ 6
- `parentesco`: Jefe / Conyuge / Hijo / Otro_familiar
- `lengua_materna`: Castellano / Quechua / Aimara / Otra_nativa / Extranjera_otra
- `campo_estudio`: 8 campos de la carrera (de `p301a1`) + Sin_carrera
- `y_reg`: log1p(ingreso anual en soles)


In [ ]:
df = paso3[['p208a', 'p207', 'p203', 'p301a', 'p300a', 'p301a1',
            'dominio', 'estrato', 'ocupinf', 'i524a1']].copy()

df['area'] = (df['estrato'] <= 5).astype(int)
df['parentesco'] = df['p203'].map({1: 'Jefe', 2: 'Conyuge', 3: 'Hijo'}).fillna('Otro_familiar')

nativas = {3, 10, 11, 12, 13, 14, 15}
df['lengua_materna'] = np.where(
    df['p300a'].isin({1, 2, 4}),
    df['p300a'].map({1: 'Quechua', 2: 'Aimara', 4: 'Castellano'}),
    np.where(df['p300a'].isin(nativas), 'Otra_nativa', 'Extranjera_otra'))

MAPA_CAMPO = {1: 'Educacion', 2: 'Ciencias', 3: 'Admin_Contab_Derecho',
              4: 'Computacion_Informatica', 5: 'Ingenieria_Tecnicas',
              6: 'Agropecuaria', 7: 'Salud'}
d = pd.to_numeric(df['p301a1'], errors='coerce')
dig = (d // 100000).astype('Int64')
campo = dig.map(MAPA_CAMPO).fillna('Artes_Otras')
df['campo_estudio'] = campo.where(d.notna(), 'Sin_carrera')

df = df.rename(columns={'p208a': 'edad', 'p207': 'sexo', 'p301a': 'nivel_educ'})
df['y_reg'] = np.log1p(df['i524a1'])

# Eliminamos nulos ANTES de convertir a texto (evita errores con valores faltantes)
df = df.dropna(subset=['edad', 'sexo', 'parentesco', 'nivel_educ', 'lengua_materna',
                       'dominio', 'area']).copy()
print('Dataset final (4. Sin nulos en predictores):', df.shape)

# Convertimos las categoricas a texto para que el OneHotEncoder use las mismas
# categorias en la app de Streamlit (evita errores de tipo int/float)
df['sexo'] = df['sexo'].astype(int).astype(str)
df['nivel_educ'] = df['nivel_educ'].astype(int).astype(str)
df['dominio'] = df['dominio'].astype(int).astype(str)
df['area'] = df['area'].astype(str)
df = df.dropna(subset=['edad', 'sexo', 'parentesco', 'nivel_educ', 'lengua_materna',
                       'dominio', 'area']).copy()
print('Dataset final (4. Sin nulos en predictores):', df.shape)


## Exploración descriptiva

Distribución de las variables numéricas, frecuencias de las categóricas y correlación con el target.

In [ ]:
print(df[['edad', 'i524a1', 'y_reg']].describe().round(2))
print()
print('Porcentaje por categoría:')
for col in ['sexo', 'parentesco', 'nivel_educ', 'lengua_materna', 'dominio', 'area', 'campo_estudio']:
    print('---', col)
    print(df[col].value_counts(normalize=True).round(3).to_string())
    print()
print('Nulos totales:', int(df.isna().sum().sum()))

In [ ]:
# Correlación entre las numéricas
correl = df[['edad', 'nivel_educ', 'y_reg']].corr(method='pearson')
print(correl)
plt.figure(figsize=(6, 5))
sns.heatmap(correl, annot=True, fmt='.2f', cmap='coolwarm', square=True)
plt.title('Matriz de correlación')
plt.show()

## 3. Pre Procesamiento de datos

Separamos las variables numéricas y categóricas y creamos el `ColumnTransformer` (OneHotEncoder para categóricas, StandardScaler para numéricas). **OJO: ninguna variable de empleo/ingreso entra como predictor.**

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

# Variables de X (solo sociodemográficas + campo de estudio)
numericas = ['edad']
categoricas = ['sexo', 'parentesco', 'nivel_educ', 'lengua_materna', 'dominio', 'area', 'campo_estudio']

preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numericas),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categoricas)
])

## 4. Data X e Y / Train y Test

In [ ]:
X = df[numericas + categoricas]
y = df['y_reg']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=123)
print('X_train:', X_train.shape, '| X_test:', X_test.shape)

## 5. Pipeline

Creamos dos pipelines (Random Forest y Gradient Boosting) con el preprocesador incluido. Así el modelo guardado contiene TODAS las transformaciones.

In [ ]:
pipeline_rf = Pipeline([
    ('preproc', preprocessor),
    ('regressor', RandomForestRegressor(random_state=123))
])

pipeline_gb = Pipeline([
    ('preproc', preprocessor),
    ('regressor', GradientBoostingRegressor(random_state=123))
])

## 6. Tunning (GridSearchCV)

Buscamos los mejores hiperparámetros con validación cruzada de 5 folds para cada modelo.

In [ ]:
rf_grid = {
    'regressor__n_estimators': [100, 300],
    'regressor__max_depth': [10, 20],
    'regressor__min_samples_leaf': [2, 5]
}

gb_grid = {
    'regressor__n_estimators': [100, 300],
    'regressor__learning_rate': [0.05, 0.1],
    'regressor__max_depth': [3, 5]
}

In [ ]:
rf_tunned = GridSearchCV(pipeline_rf, rf_grid, cv=5, n_jobs=-1)
rf_tunned.fit(X_train, y_train)
print('MEJORES PARÁMETROS RANDOM FOREST:')
print(rf_tunned.best_params_)

In [ ]:
gb_tunned = GridSearchCV(pipeline_gb, gb_grid, cv=5, n_jobs=-1)
gb_tunned.fit(X_train, y_train)
print('MEJORES PARÁMETROS GRADIENT BOOSTING:')
print(gb_tunned.best_params_)

## 7. Métricas (train vs test)

RMSE, MAE y MAPE en escala log y des-transformados a soles (expm1). Las métricas en soles corresponden al ingreso ANUAL del trabajo principal (`i524a1`).


In [ ]:
def metricas_reg(y_true, y_pred):
    yt_s = np.expm1(y_true)
    yp_s = np.expm1(y_pred)
    return {'RMSE_log': mean_squared_error(y_true, y_pred) ** 0.5,
            'MAE_log': mean_absolute_error(y_true, y_pred),
            'R2_log': 1 - mean_squared_error(y_true, y_pred) / y_true.var(),
            'RMSE_soles': mean_squared_error(yt_s, yp_s) ** 0.5,
            'MAPE_soles': mean_absolute_percentage_error(yt_s, yp_s)}

resultados = pd.DataFrame({
    'Random Forest': metricas_reg(y_test, rf_tunned.predict(X_test)),
    'Gradient Boosting': metricas_reg(y_test, gb_tunned.predict(X_test)),
})
resultados.round(4)

In [ ]:
# Train vs Test (comparación por overfitting) del modelo ganador
mejor = gb_tunned if resultados.loc['MAPE_soles', 'Gradient Boosting'] < resultados.loc['MAPE_soles', 'Random Forest'] else rf_tunned
print('MODELO GANADOR:', 'Gradient Boosting' if mejor is gb_tunned else 'Random Forest')
print('(métricas en soles = ingreso anual del trabajo principal)')
print()
print('TRAIN:')
print(pd.Series(metricas_reg(y_train, mejor.predict(X_train))).round(4))
print()
print('TEST:')
print(pd.Series(metricas_reg(y_test, mejor.predict(X_test))).round(4))


## 8. Importancia de variables

Importancia por permutación del modelo ganador.

In [ ]:
from sklearn.inspection import permutation_importance

importancia = permutation_importance(estimator=mejor, X=X_train, y=y_train,
                                     n_repeats=5, scoring='neg_root_mean_squared_error',
                                     random_state=123)
df_importancia = pd.DataFrame({'importances_mean': importancia['importances_mean'],
                               'importances_std': importancia['importances_std']})
df_importancia['feature'] = X_train.columns
df_importancia = df_importancia.sort_values('importances_mean', ascending=True)

fig, ax = plt.subplots(figsize=(4, 5))
ax.barh(df_importancia['feature'], df_importancia['importances_mean'],
        xerr=df_importancia['importances_std'], align='center', alpha=0)
ax.plot(df_importancia['importances_mean'], df_importancia['feature'],
        marker='D', linestyle='', alpha=0.8, color='r')
ax.set_title('Importancia de los predictores (train)')
ax.set_xlabel('Incremento del error tras la permutación')
plt.tight_layout()
plt.show()

## 9. Despliegue

Guardamos el pipeline ganador (con todas las transformaciones incluidas), lo cargamos y probamos con una observación nueva.

In [ ]:
# Guardar el modelo
joblib.dump(mejor, 'regresor_ingreso.joblib')

# Cargar el modelo
regressor = joblib.load('regresor_ingreso.joblib')
print('Modelo guardado y cargado correctamente.')

In [ ]:
# Nueva observación (una persona asalariada de ejemplo)
obs = pd.DataFrame([{
    'edad': 30, 'sexo': '2', 'parentesco': 'Hijo', 'nivel_educ': '6',
    'lengua_materna': 'Castellano', 'dominio': '8', 'area': '1',
    'campo_estudio': 'Sin_carrera'
}])
obs


In [ ]:
# El pipeline contiene todas las transformaciones: solo hay que predecir
pred_log = regressor.predict(obs)
pred_anual = np.expm1(pred_log)
print('Ingreso anual predicho: S/', f'{pred_anual[0]:,.2f}')


### Descargar el modelo para el despliegue en Streamlit

In [ ]:
from google.colab import files
files.download('regresor_ingreso.joblib')
print('Fin del notebook.')